# NB10 — Two-Stage Transition-Aware Return Prediction

**Hypothesis under test:** the hierarchy doesn't directly predict price, but *predicted movement through behavioral-state space* predicts return distribution.

**Pipeline.** Within each walk-forward fold, split the training slice 50/50 chronologically:

1. **Stage 1** trains on the first half. `f1: (x_t, soft_membership_t) → P(z_{t+1h})` — a RF multiclass classifier predicting the next-hour hierarchy state. This is exactly the NB06/NB07 "rf_joint" model, retimed to hourly resolution and K=4 leaves.
2. **Transition features** are derived from `f1`'s probability output: the K-vector of next-state probabilities, the Shannon entropy of that distribution, the instability `1 − P(z_{t+1h} = z_t)`, the expected-state-index drift, the 2D drift in (greed, fear) leaf-center space, and the top-1-vs-top-2 margin.
3. **Stage 2** trains on the second half. Three RF models compete on log-return prediction:
   - `baseline_rf` — baseline price/volume features only.
   - `hierarchy_rf` — baseline + soft_membership_t (the NB08/NB09 hierarchy_rf, re-trained here on the second half for fair comparison).
   - `transition_rf` — baseline + soft_membership_t + transition features (the hypothesis).

All three Stage-2 models train on exactly the same rows (the second half of train), so any improvement of `transition_rf` over `hierarchy_rf` comes from the transition features alone.

**Walk-forward:** 5 expanding-window folds on BTC-USD hourly, identical fold boundaries to NB08/09. Single ticker for design validation; multi-ticker version after.

**Pass criteria:**

1. `transition_rf` IC > `hierarchy_rf` IC on ≥3 of 5 folds.
2. Mean `transition_rf − hierarchy_rf` IC lift > +0.005.
3. `transition_rf` balanced accuracy > `hierarchy_rf` balanced accuracy on ≥3 of 5 folds.
4. Permutation importance on `transition_rf` regressor shows at least one transition feature with positive importance ranking above all soft-membership leaves.


In [1]:
import math, os, json, random
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, balanced_accuracy_score, roc_auc_score,
)
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED); np.random.seed(SEED)

TICKER = "BTC-USD"
PERIOD = "720d"
INTERVAL = "1h"

HORIZON = 1
N_FOLDS = 5
MIN_TRAIN_FRACTION = 0.40
TEST_FRACTION_PER_FOLD = 0.10
F1_F2_SPLIT = 0.5            # within each fold's train, first 50% for Stage 1, last 50% for Stage 2

MAX_HIERARCHY_DEPTH = 2
MIN_LEAF_COUNT = 200

RF_N_ESTIMATORS = 200
STAGE1_PARAMS = {"max_depth": 8, "min_samples_leaf": 20}
STAGE2_PARAMS = {"max_depth": 8, "min_samples_leaf": 20}

JOURNAL_PATH = "EXPERIMENTS.md"


## Reused helpers (same as NB09)


In [2]:
def load_yfinance_hourly(ticker, period, interval):
    raw = yf.download(tickers=ticker, period=period, interval=interval,
                     auto_adjust=True, progress=False, group_by="column")
    if raw.empty: raise RuntimeError(f"No data for {ticker}")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [c[0].lower() for c in raw.columns]
    else:
        raw.columns = [str(c).lower() for c in raw.columns]
    df = raw[["open","high","low","close","volume"]].copy().reset_index()
    ts_col = [c for c in df.columns if c not in ("open","high","low","close","volume")][0]
    df = df.rename(columns={ts_col: "ts"})
    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    return df.dropna().sort_values("ts").reset_index(drop=True)


def rolling_zscore(x, w, eps=1e-8):
    m = x.rolling(w, min_periods=max(3, w // 3)).mean()
    s = x.rolling(w, min_periods=max(3, w // 3)).std()
    return (x - m) / (s + eps)


def sigmoid_np(x): return 1.0 / (1.0 + np.exp(-x))


def make_hourly_features(df, horizon):
    out = df.copy()
    for lag in (1, 2, 4, 12, 24, 168):
        out[f"ret_{lag}"] = np.log(out["close"] / out["close"].shift(lag))
    out["range_pct"] = (out["high"] - out["low"]) / out["close"]
    out["open_close_pct"] = (out["close"] - out["open"]) / out["open"]
    for w in (24, 168, 720):
        out[f"vol_{w}"] = out["ret_1"].rolling(w).std()
    for w in (24, 168, 720):
        out[f"ma_{w}"] = out["close"].rolling(w).mean()
        out[f"price_to_ma_{w}"] = out["close"] / out[f"ma_{w}"] - 1.0
    out["volume_log"] = np.log1p(out["volume"])
    out["volume_z_168"] = rolling_zscore(out["volume_log"], 168)
    for w in (168, 720):
        out[f"rolling_max_{w}"] = out["close"].rolling(w).max()
        out[f"drawdown_{w}"] = out["close"] / out[f"rolling_max_{w}"] - 1.0
    out["trend_pressure"] = rolling_zscore(out["ret_168"], 720)
    out["fear_pressure"] = (-4.0*out["drawdown_168"] - 2.0*out["drawdown_720"]
                              + 8.0*out["vol_168"] + 0.35*out["volume_z_168"]
                              - 0.50*out["ret_24"])
    out["greed_pressure"] = (1.50*out["trend_pressure"] + 2.00*out["price_to_ma_168"]
                              + 1.00*out["ret_168"] + 0.20*out["volume_z_168"]
                              - 3.00*out["vol_720"])
    out["fear_index"] = sigmoid_np(out["fear_pressure"].clip(-8, 8))
    out["greed_index"] = sigmoid_np(out["greed_pressure"].clip(-8, 8))
    out["future_log_return"] = np.log(out["close"].shift(-horizon) / out["close"])
    out["future_direction"] = (out["future_log_return"] > 0).astype(int)
    return out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


BASELINE_FEATURES = [
    "ret_1", "ret_2", "ret_4", "ret_12", "ret_24", "ret_168",
    "range_pct", "open_close_pct",
    "vol_24", "vol_168", "vol_720",
    "price_to_ma_24", "price_to_ma_168", "price_to_ma_720",
    "volume_z_168", "drawdown_168", "drawdown_720",
]


@dataclass
class MeasureNode:
    node_id: str; depth: int
    bounds: Dict[str, Tuple[float, float]]
    mass: float; count: int
    mean: Dict[str, float]
    children: Optional[List["MeasureNode"]] = None
    @property
    def is_leaf(self): return not self.children


def in_bounds(row, bounds):
    for col, (lo, hi) in bounds.items():
        v = float(row[col])
        if not (lo <= v < hi): return False
    return True


def split_node(n, sc, sv):
    lo, hi = n.bounds[sc]
    lb, rb = dict(n.bounds), dict(n.bounds)
    lb[sc] = (lo, sv); rb[sc] = (sv, hi)
    n.children = [
        MeasureNode(n.node_id+"L", n.depth+1, lb, n.mass/2, 0, {}, None),
        MeasureNode(n.node_id+"R", n.depth+1, rb, n.mass/2, 0, {}, None),
    ]
    return n.children


def collect_leaves(n):
    if n.is_leaf: return [n]
    return [l for c in n.children for l in collect_leaves(c)]


def build_hierarchy(train_df, max_depth, min_count):
    feat = ["greed_index", "fear_index"]
    root = MeasureNode("root", 0,
                       {"greed_index":(0.0,1.000001), "fear_index":(0.0,1.000001)},
                       1.0, len(train_df),
                       {c: float(train_df[c].mean()) for c in feat}, None)
    queue = [root]
    while queue:
        n = queue.pop(0)
        if n.depth >= max_depth: continue
        mask = train_df.apply(lambda r: in_bounds(r, n.bounds), axis=1)
        sub = train_df[mask]
        n.count = len(sub); n.mass = n.count / max(len(train_df), 1)
        if n.count < min_count: continue
        v = {c: float(sub[c].var()) for c in feat}
        sc = max(v, key=v.get); sv = float(sub[sc].median())
        if n.bounds[sc][0] < sv < n.bounds[sc][1]:
            queue.extend(split_node(n, sc, sv))
    for leaf in collect_leaves(root):
        mask = train_df.apply(lambda r: in_bounds(r, leaf.bounds), axis=1)
        sub = train_df[mask]
        leaf.count = len(sub); leaf.mass = leaf.count / max(len(train_df), 1)
        leaf.mean = {c: (float(sub[c].mean()) if len(sub) else 0.0) for c in feat}
    return root


def soft_membership(row, leaves, temperature=0.04):
    p = np.array([float(row["greed_index"]), float(row["fear_index"])])
    centers = np.array([[(l.bounds["greed_index"][0]+l.bounds["greed_index"][1])/2,
                          (l.bounds["fear_index"][0]+l.bounds["fear_index"][1])/2]
                         for l in leaves])
    d = np.sum((centers - p[None,:])**2, axis=1)
    logits = -d / max(temperature, 1e-8); logits -= logits.max()
    w = np.exp(logits); return w / w.sum()


def add_memberships(frame, leaves, cols):
    M = np.vstack([soft_membership(r, leaves) for _, r in frame.iterrows()])
    return pd.concat([frame.reset_index(drop=True),
                      pd.DataFrame(M, columns=cols).reset_index(drop=True)], axis=1)


def assign_leaf_index(row, leaves):
    for i, leaf in enumerate(leaves):
        if in_bounds(row, leaf.bounds): return i
    return -1


def make_folds(n_rows, n_folds, min_train_fraction, test_fraction_per_fold):
    folds = []
    for i in range(n_folds):
        train_end = int(n_rows * (min_train_fraction + i * test_fraction_per_fold))
        test_end = int(n_rows * (min_train_fraction + (i + 1) * test_fraction_per_fold))
        if test_end > n_rows: test_end = n_rows
        if test_end - train_end < 50: break
        folds.append((train_end, test_end))
    return folds


def ic(y_true, y_pred):
    if np.std(y_pred) == 0: return 0.0
    rho, _ = spearmanr(y_true, y_pred)
    return float(rho) if not np.isnan(rho) else 0.0


## Stage-1 model + transition feature derivation

`f1` is a multiclass RF predicting `z_{t+1h}` from `(baseline_features, soft_membership_t)`. The derived features per row:

- `trans_prob_0..K-1` — the K-vector of next-state probabilities (the `transition_logits`).
- `trans_entropy` — Shannon entropy of that distribution; high = uncertain regime change.
- `trans_instability` — `1 − P(z_{t+1h} = z_t)`; the probability of leaving the current state.
- `trans_delta_index` — `E[state_{t+1h}] − state_t` treating states as 1-D indices (the literal version of your `transition_delta`).
- `trans_drift_greed`, `trans_drift_fear` — `E[leaf_center_{t+1h}] − leaf_center_t` in 2D (greed, fear) space. This is the *behavioral-space velocity vector*.
- `trans_margin` — top-1 minus top-2 probability; high = confident bet on a specific next state.
- `trans_state_change` — `1` if `argmax_u P(z_{t+1h}=u) ≠ z_t`; the model's binary call on whether the regime will flip.


In [3]:
def derive_transition_features(probs, state_t, leaf_centers, K):
    eps = 1e-12
    entropy_vals = -np.sum(probs * np.log(probs + eps), axis=1)
    p_same = probs[np.arange(len(probs)), state_t]
    instability = 1.0 - p_same
    state_indices = np.arange(K)
    expected_state_idx = probs @ state_indices
    delta_index = expected_state_idx - state_t.astype(float)
    expected_center = probs @ leaf_centers              # shape (n, 2)
    current_center = leaf_centers[state_t]              # shape (n, 2)
    drift = expected_center - current_center
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    margin = sorted_probs[:, 0] - sorted_probs[:, 1]
    argmax_state = probs.argmax(axis=1)
    state_change = (argmax_state != state_t).astype(int)
    out = {f"trans_prob_{i}": probs[:, i] for i in range(K)}
    out.update({
        "trans_entropy": entropy_vals,
        "trans_instability": instability,
        "trans_delta_index": delta_index,
        "trans_drift_greed": drift[:, 0],
        "trans_drift_fear": drift[:, 1],
        "trans_margin": margin,
        "trans_state_change": state_change,
    })
    return pd.DataFrame(out)


TRANSITION_FEATURE_NAMES_TEMPLATE = (
    lambda K: [f"trans_prob_{i}" for i in range(K)] + [
        "trans_entropy", "trans_instability", "trans_delta_index",
        "trans_drift_greed", "trans_drift_fear", "trans_margin", "trans_state_change",
    ]
)


## Per-fold runner


In [4]:
def _rf_reg(params=STAGE2_PARAMS):
    return RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1, **params)

def _rf_cls(params=STAGE2_PARAMS):
    return RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1,
                                   class_weight="balanced_subsample", **params)


def run_one_fold(df_feat, fold_idx, train_end, test_end):
    train = df_feat.iloc[:train_end].copy()
    test = df_feat.iloc[train_end:test_end].copy()

    # --- Build hierarchy from full train slice ---
    root = build_hierarchy(train, MAX_HIERARCHY_DEPTH, MIN_LEAF_COUNT)
    leaves = collect_leaves(root)
    K = len(leaves)
    leaf_cols = [f"leaf_{l.node_id}" for l in leaves]
    leaf_centers = np.array([[(l.bounds["greed_index"][0]+l.bounds["greed_index"][1])/2,
                                (l.bounds["fear_index"][0]+l.bounds["fear_index"][1])/2]
                               for l in leaves])

    train_m = add_memberships(train, leaves, leaf_cols)
    test_m = add_memberships(test, leaves, leaf_cols)

    # Discrete state assignments
    train_m["state_t"] = train_m.apply(lambda r: assign_leaf_index(r, leaves), axis=1)
    test_m["state_t"]  = test_m.apply(lambda r: assign_leaf_index(r, leaves), axis=1)

    # Future state target for Stage 1 (shifted only inside train_m; test_m doesn't need it for prediction)
    train_m["state_t_future"] = train_m["state_t"].shift(-HORIZON)
    train_m = train_m.dropna(subset=["state_t_future"]).reset_index(drop=True)
    train_m["state_t_future"] = train_m["state_t_future"].astype(int)
    train_m["state_t"] = train_m["state_t"].astype(int)
    test_m["state_t"] = test_m["state_t"].astype(int)

    # --- Stage 1 / Stage 2 chronological split ---
    mid = int(len(train_m) * F1_F2_SPLIT)
    s1_train = train_m.iloc[:mid].copy()
    s2_train = train_m.iloc[mid:].copy()

    # --- Stage 1: f1 maps (baseline + soft_membership) -> state_t_future ---
    s1_features = BASELINE_FEATURES + leaf_cols
    f1 = _rf_cls(STAGE1_PARAMS)
    f1.fit(s1_train[s1_features], s1_train["state_t_future"])

    # Apply f1 to s2_train and test to get transition feature inputs
    s2_train_probs = f1.predict_proba(s2_train[s1_features])
    test_probs = f1.predict_proba(test_m[s1_features])

    # f1 might not have seen all K classes during training; handle missing columns
    classes_seen = list(f1.classes_)
    full_probs = lambda P: pd.DataFrame(P, columns=[f"_c{c}" for c in classes_seen]).reindex(
        columns=[f"_c{c}" for c in range(K)], fill_value=0.0
    ).to_numpy()
    s2_train_probs_full = full_probs(s2_train_probs)
    test_probs_full = full_probs(test_probs)

    s2_train_trans = derive_transition_features(
        s2_train_probs_full, s2_train["state_t"].to_numpy(), leaf_centers, K
    )
    test_trans = derive_transition_features(
        test_probs_full, test_m["state_t"].to_numpy(), leaf_centers, K
    )

    s2_train = pd.concat([s2_train.reset_index(drop=True), s2_train_trans.reset_index(drop=True)], axis=1)
    test_m = pd.concat([test_m.reset_index(drop=True), test_trans.reset_index(drop=True)], axis=1)

    trans_cols = TRANSITION_FEATURE_NAMES_TEMPLATE(K)
    feature_sets = [
        ("baseline_rf", BASELINE_FEATURES),
        ("hierarchy_rf", BASELINE_FEATURES + leaf_cols),
        ("transition_rf", BASELINE_FEATURES + leaf_cols + trans_cols),
    ]

    y_r = test_m["future_log_return"].to_numpy()
    y_c = test_m["future_direction"].to_numpy().astype(int)

    rows = []
    models = {}
    for name, fcols in feature_sets:
        reg = _rf_reg(); reg.fit(s2_train[fcols], s2_train["future_log_return"])
        cls = _rf_cls(); cls.fit(s2_train[fcols], s2_train["future_direction"].astype(int))
        pred_r = reg.predict(test_m[fcols])
        pred_c = cls.predict(test_m[fcols])
        prob_c = cls.predict_proba(test_m[fcols])[:, 1] if 1 in cls.classes_ else np.full(len(y_c), 0.5)
        rows.append({
            "fold": fold_idx, "model": name, "kind": "regression",
            "mae": float(mean_absolute_error(y_r, pred_r)),
            "rmse": float(math.sqrt(mean_squared_error(y_r, pred_r))),
            "ic": ic(y_r, pred_r), "n_test": len(y_r),
        })
        rows.append({
            "fold": fold_idx, "model": name, "kind": "classification",
            "accuracy": float(accuracy_score(y_c, pred_c)),
            "balanced_accuracy": float(balanced_accuracy_score(y_c, pred_c)),
            "roc_auc": float(roc_auc_score(y_c, prob_c)) if len(np.unique(y_c)) == 2 else float("nan"),
            "n_test": len(y_c),
        })
        models[name] = (reg, cls, fcols)

    # Trivial baselines
    pr_zero = np.zeros_like(y_r)
    pr_pers = test_m["ret_1"].to_numpy()
    majority = int(s2_train["future_direction"].mean() >= 0.5)
    pc_majority = np.full_like(y_c, majority)
    pc_pers = (pr_pers > 0).astype(int)
    for name, pr in [("zero", pr_zero), ("persistence", pr_pers)]:
        rows.append({
            "fold": fold_idx, "model": name, "kind": "regression",
            "mae": float(mean_absolute_error(y_r, pr)),
            "rmse": float(math.sqrt(mean_squared_error(y_r, pr))),
            "ic": ic(y_r, pr), "n_test": len(y_r),
        })
    for name, pc in [("majority", pc_majority), ("persistence", pc_pers)]:
        rows.append({
            "fold": fold_idx, "model": name, "kind": "classification",
            "accuracy": float(accuracy_score(y_c, pc)),
            "balanced_accuracy": float(balanced_accuracy_score(y_c, pc)),
            "roc_auc": float("nan"),
            "n_test": len(y_c),
        })

    info = {"fold": fold_idx, "K": K, "trans_cols": trans_cols, "leaf_cols": leaf_cols,
            "s1_train_rows": len(s1_train), "s2_train_rows": len(s2_train),
            "f1_classes_seen": classes_seen}
    return pd.DataFrame(rows), models, test_m, info


## Run all folds


In [5]:
df_raw = load_yfinance_hourly(TICKER, PERIOD, INTERVAL)
df_feat = make_hourly_features(df_raw, HORIZON)
print(f"{TICKER}: {len(df_raw)} raw -> {len(df_feat)} bars after dropna, "
      f"{df_feat['ts'].min()} -> {df_feat['ts'].max()}")

folds = make_folds(len(df_feat), N_FOLDS, MIN_TRAIN_FRACTION, TEST_FRACTION_PER_FOLD)
print(f"{len(folds)} folds")

all_rows = []
all_models = {}
all_test = {}
all_info = {}
for i, (tr, te) in enumerate(folds):
    print(f"fold={i} train_end={tr} test_end={te}")
    rdf, models, test_m, info = run_one_fold(df_feat, i, tr, te)
    all_rows.append(rdf)
    all_models[i] = models
    all_test[i] = test_m
    all_info[i] = info

agg = pd.concat(all_rows, ignore_index=True)
agg[["fold", "kind", "model", "mae", "rmse", "ic", "accuracy", "balanced_accuracy", "roc_auc", "n_test"]]


BTC-USD: 17100 raw -> 16379 bars after dropna, 2024-06-22 00:00:00+00:00 -> 2026-05-12 11:00:00+00:00
5 folds
fold=0 train_end=6551 test_end=8189


fold=1 train_end=8189 test_end=9827


fold=2 train_end=9827 test_end=11465


fold=3 train_end=11465 test_end=13103


fold=4 train_end=13103 test_end=14741


,fold,kind,model,mae,rmse,ic,accuracy,balanced_accuracy,roc_auc,n_test
0,0,regression,baseline_rf,0.003213,0.004796,0.004473,NaN,NaN,NaN,1638
1,0,classification,baseline_rf,NaN,NaN,NaN,0.506105,0.505797,0.522382,1638
2,0,regression,hierarchy_rf,0.003210,0.004791,0.010389,NaN,NaN,NaN,1638
3,0,classification,hierarchy_rf,NaN,NaN,NaN,0.515263,0.515056,0.523825,1638
4,0,regression,transition_rf,0.003196,0.004779,0.016904,NaN,NaN,NaN,1638
5,0,classification,transition_rf,NaN,NaN,NaN,0.519536,0.519820,0.537047,1638
6,0,regression,zero,0.003121,0.004713,0.000000,NaN,NaN,NaN,1638
7,0,regression,persistence,0.004614,0.006815,-0.042573,NaN,NaN,NaN,1638
8,0,classification,majority,NaN,NaN,NaN,0.505495,0.500000,NaN,1638
9,0,classification,persistence,NaN,NaN,NaN,0.493284,0.493223,NaN,1638


## Aggregate across folds + lift tables


In [6]:
reg = agg[agg["kind"] == "regression"]
cls = agg[agg["kind"] == "classification"]

reg_summary = reg.groupby("model")[["mae", "rmse", "ic"]].agg(["mean", "std"]).round(5)
cls_summary = cls.groupby("model")[["accuracy", "balanced_accuracy", "roc_auc"]].agg(["mean", "std"]).round(5)

print("=== Regression summary (mean ± std across folds) ===")
print(reg_summary)
print()
print("=== Classification summary (mean ± std across folds) ===")
print(cls_summary)


=== Regression summary (mean ± std across folds) ===
                   mae              rmse                ic         
                  mean      std     mean      std     mean      std
model                                                              
baseline_rf    0.00299  0.00063  0.00468  0.00123  0.02022  0.01055
hierarchy_rf   0.00299  0.00063  0.00469  0.00123  0.02752  0.01279
persistence    0.00430  0.00089  0.00656  0.00167 -0.01849  0.02707
transition_rf  0.00298  0.00063  0.00468  0.00123  0.02806  0.00740
zero           0.00296  0.00061  0.00464  0.00121  0.00000  0.00000

=== Classification summary (mean ± std across folds) ===
              accuracy          balanced_accuracy           roc_auc         
                  mean      std              mean      std     mean      std
model                                                                       
baseline_rf    0.51697  0.01350           0.51856  0.01351  0.52045  0.00568
hierarchy_rf   0.51795  0.00547      

In [7]:
# Per-fold tables and lift vs hierarchy
ic_pivot = reg[reg["model"].isin(["baseline_rf", "hierarchy_rf", "transition_rf"])].pivot_table(
    index="fold", columns="model", values="ic"
)
ba_pivot = cls[cls["model"].isin(["baseline_rf", "hierarchy_rf", "transition_rf"])].pivot_table(
    index="fold", columns="model", values="balanced_accuracy"
)
auc_pivot = cls[cls["model"].isin(["baseline_rf", "hierarchy_rf", "transition_rf"])].pivot_table(
    index="fold", columns="model", values="roc_auc"
)

print("=== Per-fold IC (1h horizon) ===")
print(ic_pivot.round(5))
print()
print("=== Per-fold balanced accuracy ===")
print(ba_pivot.round(4))
print()
print("=== Per-fold ROC AUC ===")
print(auc_pivot.round(4))
print()
print("--- Lifts: transition_rf − hierarchy_rf ---")
print(pd.DataFrame({
    "ic_lift": ic_pivot["transition_rf"] - ic_pivot["hierarchy_rf"],
    "bal_acc_lift_pp": (ba_pivot["transition_rf"] - ba_pivot["hierarchy_rf"]) * 100,
    "auc_lift": auc_pivot["transition_rf"] - auc_pivot["hierarchy_rf"],
}).round(5))


=== Per-fold IC (1h horizon) ===
model  baseline_rf  hierarchy_rf  transition_rf
fold                                           
0          0.00447       0.01039        0.01690
1          0.01755       0.04050        0.03247
2          0.02952       0.02920        0.02399
3          0.01921       0.01911        0.03350
4          0.03033       0.03841        0.03343

=== Per-fold balanced accuracy ===
model  baseline_rf  hierarchy_rf  transition_rf
fold                                           
0           0.5058        0.5151         0.5198
1           0.5156        0.5249         0.5259
2           0.5416        0.5142         0.5106
3           0.5147        0.5260         0.5072
4           0.5151        0.5157         0.5139

=== Per-fold ROC AUC ===
model  baseline_rf  hierarchy_rf  transition_rf
fold                                           
0           0.5224        0.5238         0.5370
1           0.5177        0.5270         0.5358
2           0.5290        0.5275         

## Pass-fail summary


In [8]:
ic_lift = ic_pivot["transition_rf"] - ic_pivot["hierarchy_rf"]
ba_lift = ba_pivot["transition_rf"] - ba_pivot["hierarchy_rf"]
auc_lift = auc_pivot["transition_rf"] - auc_pivot["hierarchy_rf"]

print(f"transition_rf vs hierarchy_rf IC:")
print(f"  mean lift = {ic_lift.mean():+.5f}")
print(f"  folds with positive IC lift: {int((ic_lift > 0).sum())}/{len(ic_lift)}")
print(f"  pass IC criteria (lift>+0.005 AND >=3/5 folds positive): "
      f"{ic_lift.mean() > 0.005 and (ic_lift > 0).sum() >= 3}")
print()
print(f"transition_rf vs hierarchy_rf balanced accuracy:")
print(f"  mean lift = {ba_lift.mean() * 100:+.2f} pp")
print(f"  folds with positive bal_acc lift: {int((ba_lift > 0).sum())}/{len(ba_lift)}")
print(f"  pass bal_acc criterion (>=3/5 positive): {(ba_lift > 0).sum() >= 3}")
print()
print(f"transition_rf vs hierarchy_rf ROC AUC:")
print(f"  mean lift = {auc_lift.mean():+.5f}")
print(f"  folds with positive AUC lift: {int((auc_lift > 0).sum())}/{len(auc_lift)}")


transition_rf vs hierarchy_rf IC:
  mean lift = +0.00054
  folds with positive IC lift: 2/5
  pass IC criteria (lift>+0.005 AND >=3/5 folds positive): False

transition_rf vs hierarchy_rf balanced accuracy:
  mean lift = -0.37 pp
  folds with positive bal_acc lift: 2/5
  pass bal_acc criterion (>=3/5 positive): False

transition_rf vs hierarchy_rf ROC AUC:
  mean lift = +0.00405
  folds with positive AUC lift: 3/5


## Permutation importance on transition_rf regressor (pooled across folds)

For each fold's `transition_rf` regression model, compute permutation importance on the fold's test set (scoring `neg_mean_absolute_error`), then average across folds. The question: do any transition features rank above the soft-membership leaves in held-out importance?


In [9]:
def perm_imp_for_model(model, x_test, y_test, feature_columns, scoring, n_repeats=5):
    pi = permutation_importance(model, x_test, y_test, n_repeats=n_repeats,
                                random_state=SEED, n_jobs=-1, scoring=scoring)
    return pd.DataFrame({"feature": feature_columns,
                         "perm_importance_mean": pi.importances_mean,
                         "perm_importance_std": pi.importances_std})


fold_imps = []
for i in range(len(folds)):
    reg, _, fcols = all_models[i]["transition_rf"]
    test_m = all_test[i]
    pim = perm_imp_for_model(reg, test_m[fcols], test_m["future_log_return"].to_numpy(),
                             fcols, scoring="neg_mean_absolute_error")
    pim["fold"] = i
    fold_imps.append(pim)

perm_df = pd.concat(fold_imps, ignore_index=True)
avg_imp = perm_df.groupby("feature")["perm_importance_mean"].agg(["mean", "std"]).sort_values("mean", ascending=False)
print("=== Permutation importance, transition_rf regressor (mean over 5 folds) ===")
print(avg_imp.round(5).head(40))

# Classify feature family
def family(name):
    if name.startswith("leaf_"): return "soft_membership"
    if name.startswith("trans_"): return "transition_derived"
    return "baseline"
avg_imp_family = avg_imp.reset_index()
avg_imp_family["family"] = avg_imp_family["feature"].map(family)
print()
print("=== Aggregate by family ===")
print(avg_imp_family.groupby("family")[["mean"]].agg(["mean", "sum", "count"]).round(5))


=== Permutation importance, transition_rf regressor (mean over 5 folds) ===
                    mean      std
feature                          
drawdown_720         0.0  0.00001
trans_prob_0         0.0  0.00001
ret_24               0.0  0.00000
ret_4                0.0  0.00000
ret_168              0.0  0.00000
vol_24               0.0  0.00000
vol_168              0.0  0.00000
trans_prob_3         0.0  0.00000
open_close_pct       0.0  0.00000
ret_1                0.0  0.00000
trans_delta_index    0.0  0.00000
leaf_rootLR          0.0  0.00000
trans_drift_greed    0.0  0.00000
trans_prob_2         0.0  0.00000
trans_state_change   0.0  0.00000
trans_drift_fear    -0.0  0.00000
price_to_ma_24      -0.0  0.00000
trans_entropy       -0.0  0.00000
leaf_rootRR         -0.0  0.00000
leaf_rootLL         -0.0  0.00000
trans_prob_1        -0.0  0.00000
volume_z_168        -0.0  0.00000
drawdown_168        -0.0  0.00000
trans_instability   -0.0  0.00000
vol_720             -0.0  0.00000
trans_

## Append to EXPERIMENTS.md


In [10]:
def append_to_journal(agg_df, ic_pivot, ba_pivot, auc_pivot, journal_path=JOURNAL_PATH):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    config = {
        "notebook": "10_two_stage_transition_return.ipynb",
        "ticker": TICKER, "period": PERIOD, "interval": INTERVAL,
        "horizon_hours": HORIZON, "n_folds": N_FOLDS,
        "f1_f2_split": F1_F2_SPLIT,
        "max_hierarchy_depth": MAX_HIERARCHY_DEPTH,
        "stage1_params": STAGE1_PARAMS, "stage2_params": STAGE2_PARAMS,
        "rf_n_estimators": RF_N_ESTIMATORS, "seed": SEED,
    }
    reg_summary = agg_df[agg_df["kind"] == "regression"].groupby("model")[["mae", "rmse", "ic"]].agg(["mean", "std"]).round(5)
    cls_summary = agg_df[agg_df["kind"] == "classification"].groupby("model")[["accuracy", "balanced_accuracy", "roc_auc"]].agg(["mean", "std"]).round(5)

    ic_lift = (ic_pivot["transition_rf"] - ic_pivot["hierarchy_rf"]).round(5)
    ba_lift = ((ba_pivot["transition_rf"] - ba_pivot["hierarchy_rf"]) * 100).round(3)
    auc_lift = (auc_pivot["transition_rf"] - auc_pivot["hierarchy_rf"]).round(5)
    lift_df = pd.DataFrame({"ic_lift": ic_lift, "bal_acc_lift_pp": ba_lift, "auc_lift": auc_lift})

    parts = ["\n---\n"]
    parts.append(f"\n## {timestamp} — NB10 two-stage transition-aware return prediction (BTC-USD, 1h, K=4)\n")
    parts.append(f"\n**Config**\n\n```json\n{json.dumps(config, indent=2)}\n```\n")
    parts.append("\n**Per-fold lifts: `transition_rf − hierarchy_rf`**\n\n")
    parts.append(lift_df.to_markdown())
    parts.append("\n\n**Regression summary (mean ± std across folds)**\n\n")
    parts.append("```\n"); parts.append(str(reg_summary)); parts.append("\n```\n")
    parts.append("\n**Classification summary (mean ± std across folds)**\n\n")
    parts.append("```\n"); parts.append(str(cls_summary)); parts.append("\n```\n")
    text = "".join(parts)
    with open(journal_path, "a") as f:
        f.write(text)
    print(f"Appended entry to {journal_path}")
    return text


journal_text = append_to_journal(agg, ic_pivot, ba_pivot, auc_pivot)


Appended entry to EXPERIMENTS.md


## What to look for

The hypothesis under test was: *predicted movement through behavioral-state space predicts return distribution.*

- **If `transition_rf` IC > `hierarchy_rf` IC on ≥3/5 folds** AND **mean lift > +0.005**, the hypothesis is supported on BTC. Next step: NB11 multi-crypto.
- **If transition features dominate the permutation-importance ranking** (e.g., `trans_drift_greed`, `trans_entropy`, `trans_instability` above all `leaf_*` columns), it's the *predicted dynamics* not the *static membership* that's carrying signal.
- **If `transition_rf ≈ hierarchy_rf`**, the transition features are redundant with the raw soft-membership — the stage-1 model is just reorganizing the same information.
- **If `transition_rf < hierarchy_rf`**, the stage-1 predictions add noise relative to ground-truth features. Either the f1 model is too weak, or the hypothesis is wrong.

Honest expectation: NB06's `rf_joint` beat persistence on state prediction by ~6 pp on BTC daily. The hourly version is a less mature task, so f1 may be weaker. If f1's predictions add even +0.003 IC over hierarchy_rf consistently across folds, that's a real win — small in absolute terms but signed in the right direction with clean architecture.
